In [ ]:
#https://www.slingacademy.com/article/implementing-multivariate-forecasting-using-grus-in-pytorch/

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import csv
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')

test_raw['OT'] = 0.0

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw[['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']])
test_scaled = scaler.transform(test_raw[['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']])


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=x.device).requires_grad_()

        # Forward pass
        out, hn = self.gru(x, h0.detach())

        # Reshape
        out = self.fc(out[:, -1, :])
        return out

training_sequences = create_sequences(train_scaled, 1)
test_sequences = create_sequences(test_scaled, 1)

# Split features (all except OT) and labels (OT)
train_features = train_scaled[:, :-1]
train_labels   = train_scaled[:,  -1]
test_features  = test_scaled[:, :-1]

# Convert to GRU-friendly shape (batch, seq_len=1, features)
tensor_train_features = torch.Tensor(train_features).unsqueeze(1).to(device)
tensor_train_labels   = torch.Tensor(train_labels).unsqueeze(1).to(device)
tensor_test           = torch.Tensor(test_features).unsqueeze(1).to(device)

# Put it into dataset
dataset = TensorDataset(tensor_train_features, tensor_train_labels)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Define model
model = GRUModel(input_dim=6, hidden_dim=64, layer_dim=2, output_dim=1).to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_model(model, dataloader, criterion, optimizer, epochs=20):
    for epoch in  range(epochs):
        for i, (features, labels) in enumerate(dataloader):
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if(i + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {loss.item():.4f}')

train_model(model, dataloader, criterion, optimizer)

model.eval()
with torch.no_grad():
    preds_scaled = model(tensor_test).cpu().numpy()

preds_full = np.zeros((preds_scaled.shape[0], train_scaled.shape[1]))
preds_full[:, -1] = preds_scaled[:, 0]

# Now inverse transform
preds_real = scaler.inverse_transform(preds_full)[:, -1]  # take only OT


# Save CSV
with open("test_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "OT"])
    for i, val in enumerate(preds_real):
        writer.writerow([i + 14000, float(val)])






Device:  cuda
Epoch [1/20], Step [10/219], Loss: 0.2781
Epoch [1/20], Step [20/219], Loss: 0.1486
Epoch [1/20], Step [30/219], Loss: 0.1387
Epoch [1/20], Step [40/219], Loss: 0.1411
Epoch [1/20], Step [50/219], Loss: 0.1404
Epoch [1/20], Step [60/219], Loss: 0.1466
Epoch [1/20], Step [70/219], Loss: 0.1247
Epoch [1/20], Step [80/219], Loss: 0.1157
Epoch [1/20], Step [90/219], Loss: 0.1191
Epoch [1/20], Step [100/219], Loss: 0.1389
Epoch [1/20], Step [110/219], Loss: 0.1254
Epoch [1/20], Step [120/219], Loss: 0.1297
Epoch [1/20], Step [130/219], Loss: 0.1375
Epoch [1/20], Step [140/219], Loss: 0.1403
Epoch [1/20], Step [150/219], Loss: 0.1529
Epoch [1/20], Step [160/219], Loss: 0.1260
Epoch [1/20], Step [170/219], Loss: 0.1471
Epoch [1/20], Step [180/219], Loss: 0.1460
Epoch [1/20], Step [190/219], Loss: 0.1139
Epoch [1/20], Step [200/219], Loss: 0.1484
Epoch [1/20], Step [210/219], Loss: 0.1333
Epoch [2/20], Step [10/219], Loss: 0.1360
Epoch [2/20], Step [20/219], Loss: 0.1315
Epoch [2